# BTS Flight Data Ingestion - Bronze Layer

This notebook implements the **Bronze layer** of the medallion architecture for US flight data from the Bureau of Transportation Statistics (BTS).

## Medallion Architecture Overview

```
RAW DATA → BRONZE (this notebook) → SILVER → GOLD
```

* **Bronze**: Raw ingestion with minimal transformation (schema enforcement, partitioning)
* **Silver**: Data quality rules, cleansing, standardization
* **Gold**: Business-level aggregations and metrics

## This Notebook's Role

**Layer**: Bronze (Raw Data Ingestion)  
**Source**: BTS TranStats On-Time Performance API  
**Output Table**: `skyops.raw.bts_ontime`  
**Output Type**: Managed Delta Table (Unity Catalog)  
**Transformation**: Minimal - schema enforcement, date normalization, partitioning  
**Time Range**: Last 4 years (~49 months)  
**Expected Rows**: ~28M  
**Runtime**: 3-4 hours (download) + 5-10 minutes (Delta table creation)

## Architecture Flow

```
BTS TranStats API
    ↓
[bts_downloader.py] ← Python script (ingestion/)
    ↓
/Volumes/skyops/raw/bts/staging/*.csv ← Staging layer (monthly CSVs)
    ↓
[THIS NOTEBOOK] ← Bronze transformation
    ↓
skyops.raw.bts_ontime ← Bronze Delta table
    ↓
[Future: silver/bts_ontime_clean.py] ← Silver layer
    ↓
[Future: gold/bts_analytics.py] ← Gold layer
```

## Folder Structure

```
/SkyOps/
├── ingestion/
│   └── bts_downloader.py        # Raw data fetcher (HTTP → CSV)
├── bronze/                       # ← YOU ARE HERE
│   └── bts_ontime_ingest.py     # Bronze ingestion (CSV → Delta)
├── silver/
│   └── (future: cleaned tables)
└── gold/
    └── (future: aggregated metrics)
```

## Catalog Structure

```
skyops (catalog)
├── raw (schema)                 # ← Bronze layer output
│   ├── bts_ontime              # This notebook creates this
│   └── bts (volume)            # Staging CSVs stored here
├── silver (schema)              # Future: cleaned data
└── gold (schema)                # Future: business metrics
```

## Execution Steps

1. **Install dependencies** (Cell 2)
2. **Download staging CSVs** (Cell 4) - 3-4 hours, resume-friendly
3. **Create Bronze Delta table** (Cell 6) - Combines CSVs with schema enforcement
4. **Validate data integrity** (Cells 7-11) - No data loss verification

## Replication Instructions

**Prerequisites:**
* Unity Catalog Volume: `skyops.raw.bts` (must exist)
* Databricks Serverless or cluster with internet access
* 4 hours of runtime for first download

**First-time setup:**
```python
# Run all cells in order (1 → 11)
# Expected: 28.2M rows in skyops.raw.bts_ontime
```

**Incremental updates:**
```python
# Modify Cell 4 parameters:
--years 1          # Last 1 year only
--month 2026-03    # Single month
--force            # Re-download existing
```

## Bronze Layer Characteristics

✓ **Raw data preservation**: Minimal transformation  
✓ **Schema enforcement**: Explicit types prevent drift  
✓ **Date normalization**: Standardized yyyyMMdd format  
✓ **Partitioning**: YEAR-based for query performance  
✓ **Idempotency**: Re-running overwrites safely  
✓ **Data lineage**: Staging CSVs preserved for audit

## Next Steps (Silver Layer)

1. Parse FL_DATE to proper date/timestamp types
2. Add timezone conversions for flight times
3. Enrich with airport/airline reference data
4. Apply data quality rules (nulls, outliers)
5. Create slowly-changing dimension (SCD) tracking

In [0]:
%pip install requests beautifulsoup4 lxml python-dateutil pyarrow --quiet
dbutils.library.restartPython()

In [0]:
import subprocess
import sys

In [0]:
# This will download 4 years of BTS flight data (~3-4 hours runtime)
# Output: Monthly CSV files in /Volumes/skyops/raw/bts/staging/
# Resume-friendly: Already-downloaded files are skipped automatically

print("Starting BTS data download...")
print("Expected runtime: 3-4 hours for 4 years (49 months)")
print("Output: /Volumes/skyops/raw/bts/staging/*.csv")
print("=" * 60)

result = subprocess.run([
    sys.executable,
    "/Workspace/Users/hrpapasani@gmail.com/SkyOps/ingestion/bts_downloader.py",
    "--years", "4",
    "--output-dir", "/Volumes/skyops/raw/bts"
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise Exception(f"Download failed with return code {result.returncode}")

print("\n" + "=" * 60)
print("✓ Download complete! Staging CSVs ready for Delta table creation")
print("=" * 60)

In [0]:
dbutils.fs.ls("/Volumes/skyops/raw/bts/staging")

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)
from pyspark.sql.functions import to_date, year, month, col

print("="*70)
print("BRONZE LAYER: Creating Delta table from staging CSVs")
print("="*70)
print("Using explicit schema to prevent inference drift\n")

# Define explicit schema (matches normalized BTS data)
schema = StructType([
    StructField("FL_DATE", StringType(), True),           # yyyyMMdd format
    StructField("AIRLINE_CODE", StringType(), True),
    StructField("DOT_CODE", IntegerType(), True),
    StructField("FL_NUMBER", IntegerType(), True),
    StructField("OriginAirportID", IntegerType(), True),
    StructField("DestAirportID", IntegerType(), True),
    StructField("ORIGIN_CITY", StringType(), True),
    StructField("DEST", StringType(), True),
    StructField("DEST_CITY", StringType(), True),
    StructField("CRS_DEP_TIME", StringType(), True),
    StructField("DEP_TIME", StringType(), True),
    StructField("DEP_DELAY", DoubleType(), True),
    StructField("TAXI_OUT", DoubleType(), True),
    StructField("WHEELS_OFF", StringType(), True),
    StructField("WHEELS_ON", StringType(), True),
    StructField("TAXI_IN", DoubleType(), True),
    StructField("CRS_ARR_TIME", StringType(), True),
    StructField("ARR_TIME", StringType(), True),
    StructField("ARR_DELAY", DoubleType(), True),
    StructField("CANCELLED", DoubleType(), True),
    StructField("CANCELLATION_CODE", StringType(), True),
    StructField("DIVERTED", DoubleType(), True),
    StructField("CRS_ELAPSED_TIME", DoubleType(), True),
    StructField("ELAPSED_TIME", DoubleType(), True),
    StructField("AIR_TIME", DoubleType(), True),
    StructField("DISTANCE", DoubleType(), True),
    StructField("DELAY_DUE_CARRIER", DoubleType(), True),
    StructField("DELAY_DUE_WEATHER", DoubleType(), True),
    StructField("DELAY_DUE_NAS", DoubleType(), True),
    StructField("DELAY_DUE_SECURITY", DoubleType(), True),
    StructField("DELAY_DUE_LATE_AIRCRAFT", DoubleType(), True),
    StructField("Tail_Number", StringType(), True),
])

# Read all staging CSVs with explicit schema
print("Reading staging CSV files...")
df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load("/Volumes/skyops/raw/bts/staging/*.csv")

print(f"Loaded DataFrame with {len(df.columns)} columns")

# Add YEAR and MONTH partition columns from FL_DATE (yyyyMMdd)
print("Adding YEAR and MONTH partition columns...")
df = df.withColumn(
    "FL_DATE_PARSED", 
    to_date(col("FL_DATE"), "yyyyMMdd")
)
df = df.withColumn("YEAR", year(col("FL_DATE_PARSED")))
df = df.withColumn("MONTH", month(col("FL_DATE_PARSED")))
df = df.drop("FL_DATE_PARSED")  # Drop temp column

# Write directly as managed Delta table (UC handles storage)
table_name = "skyops.raw.bts_ontime"
print(f"\nWriting to Bronze Delta table: {table_name}")
print("Layer: BRONZE (raw data with minimal transformation)")
print("Partitioned by YEAR for query performance...")
print("This may take a few minutes...\n")

df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("YEAR") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✓ Bronze Delta table created successfully: {table_name}")
print(f"  Layer: BRONZE (raw ingestion)")
print(f"  Partitioned by: YEAR")
print(f"  Features: ACID, time travel, schema evolution")
print(f"\nVerifying row count...")
final_count = spark.table(table_name).count()
print(f"✓ Final row count: {final_count:,}")
print(f"\n✓✓✓ Bronze layer complete! Ready for Silver transformation.")
print("="*70)

In [0]:
# =============================================================================
# DATA INTEGRITY VALIDATION - Part 1: Table Metadata
# =============================================================================

print("\n" + "="*70)
print("VALIDATION PART 1: TABLE METADATA & STRUCTURE")
print("="*70 + "\n")

# 1. Confirm table exists and is accessible
try:
    table_info = spark.sql("DESCRIBE DETAIL skyops.raw.bts_ontime").collect()[0]
    print("✓ Table exists: skyops.raw.bts_ontime")
    print(f"  Format: {table_info['format']}")
    print(f"  Location: {table_info['location']}")
    print(f"  Created at: {table_info['createdAt']}")
    print(f"  Number of files: {table_info['numFiles']}")
    print(f"  Size on disk: {table_info['sizeInBytes'] / (1024**3):.2f} GB")
except Exception as e:
    print(f"✗ ERROR: Table not found or inaccessible: {e}")
    raise

print("\n" + "-"*70)

# 2. Verify schema structure
print("\n✓ Schema Validation:")
df_schema = spark.table("skyops.raw.bts_ontime")
expected_columns = [
    "FL_DATE", "AIRLINE_CODE", "DOT_CODE", "FL_NUMBER",
    "OriginAirportID", "DestAirportID", "ORIGIN_CITY", "DEST", "DEST_CITY",
    "CRS_DEP_TIME", "DEP_TIME", "DEP_DELAY", "TAXI_OUT",
    "WHEELS_OFF", "WHEELS_ON", "TAXI_IN",
    "CRS_ARR_TIME", "ARR_TIME", "ARR_DELAY",
    "CANCELLED", "CANCELLATION_CODE", "DIVERTED",
    "CRS_ELAPSED_TIME", "ELAPSED_TIME", "AIR_TIME", "DISTANCE",
    "DELAY_DUE_CARRIER", "DELAY_DUE_WEATHER", "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY", "DELAY_DUE_LATE_AIRCRAFT",
    "Tail_Number", "YEAR", "MONTH"
]

actual_columns = df_schema.columns
print(f"  Expected columns: {len(expected_columns)}")
print(f"  Actual columns: {len(actual_columns)}")

if set(expected_columns) == set(actual_columns):
    print("  ✓ All expected columns present")
else:
    missing = set(expected_columns) - set(actual_columns)
    extra = set(actual_columns) - set(expected_columns)
    if missing:
        print(f"  ✗ Missing columns: {missing}")
    if extra:
        print(f"  ✗ Extra columns: {extra}")

print("\n" + "-"*70)

In [0]:
# =============================================================================
# DATA INTEGRITY VALIDATION - Part 2: Row Count Comparison
# =============================================================================

print("\n" + "="*70)
print("VALIDATION PART 2: ROW COUNT VERIFICATION")
print("="*70 + "\n")

# 1. Count rows in staging CSVs
print("Counting rows in staging CSVs...")
staging_path = "/Volumes/skyops/raw/bts/staging/*.csv"
staging_df = spark.read.format("csv") \
    .option("header", "true") \
    .load(staging_path)
staging_count = staging_df.count()
print(f"✓ Staging CSV total: {staging_count:,} rows")

print("\n" + "-"*70)

# 2. Count rows in Delta table
print("\nCounting rows in Delta table...")
table_df = spark.table("skyops.raw.bts_ontime")
table_count = table_df.count()
print(f"✓ Delta table total: {table_count:,} rows")

print("\n" + "-"*70)

# 3. Compare counts
print("\nRow Count Comparison:")
print(f"  Staging CSVs:  {staging_count:>12,}")
print(f"  Delta Table:   {table_count:>12,}")
print(f"  Difference:    {abs(staging_count - table_count):>12,}")

if staging_count == table_count:
    print("\n✓✓✓ PERFECT MATCH: No data loss during ingestion!")
else:
    loss_pct = abs(staging_count - table_count) / staging_count * 100
    print(f"\n✗ WARNING: Row count mismatch ({loss_pct:.4f}% difference)")
    print("  Investigate data loss or duplication")

print("\n" + "="*70)

In [0]:
# =============================================================================
# DATA INTEGRITY VALIDATION - Part 3: Data Quality Checks
# =============================================================================

from pyspark.sql.functions import col, count, countDistinct, when, isnan, isnull

print("\n" + "="*70)
print("VALIDATION PART 3: DATA QUALITY CHECKS")
print("="*70 + "\n")

df = spark.table("skyops.raw.bts_ontime")

# 1. Check for completely empty rows
print("1. Checking for completely empty rows...")
empty_check = df.filter(
    (col("FL_DATE").isNull()) & 
    (col("AIRLINE_CODE").isNull()) & 
    (col("FL_NUMBER").isNull())
)
empty_count = empty_check.count()
if empty_count == 0:
    print(f"   ✓ No completely empty rows")
else:
    print(f"   ✗ WARNING: {empty_count:,} completely empty rows found")

print("\n" + "-"*70)

# 2. Verify partition coverage (years)
print("\n2. Checking partition coverage (YEAR)...")
year_dist = df.groupBy("YEAR").count().orderBy("YEAR").collect()
print(f"   Found {len(year_dist)} distinct years:")
for row in year_dist:
    print(f"     {row['YEAR']}: {row['count']:>12,} rows")

expected_years = [2022, 2023, 2024, 2025, 2026]
actual_years = [row['YEAR'] for row in year_dist]
if set(expected_years) == set(actual_years):
    print("   ✓ All expected years present (2022-2026)")
else:
    missing_years = set(expected_years) - set(actual_years)
    if missing_years:
        print(f"   ✗ WARNING: Missing years: {missing_years}")

print("\n" + "-"*70)

# 3. Check key column null rates
print("\n3. Checking key column completeness...")
total_rows = df.count()
key_columns = ["FL_DATE", "AIRLINE_CODE", "FL_NUMBER", "ORIGIN_CITY", "DEST"]

for col_name in key_columns:
    null_count = df.filter(col(col_name).isNull()).count()
    null_pct = (null_count / total_rows) * 100
    status = "✓" if null_pct < 1 else "⚠"
    print(f"   {status} {col_name:20s}: {null_pct:6.2f}% null ({null_count:,} rows)")

print("\n" + "-"*70)

# 4. Verify date range coverage
print("\n4. Checking date range coverage...")
date_range = df.selectExpr(
    "MIN(FL_DATE) as min_date",
    "MAX(FL_DATE) as max_date",
    "COUNT(DISTINCT FL_DATE) as distinct_dates"
).collect()[0]

print(f"   Date range: {date_range['min_date']} to {date_range['max_date']}")
print(f"   Distinct dates: {date_range['distinct_dates']:,}")
print(f"   Expected ~49 months (Feb 2022 - Feb 2026)")

print("\n" + "="*70)
print("\n✓✓✓ DATA QUALITY VALIDATION COMPLETE")
print("="*70)

In [0]:
# =============================================================================
# DATA INTEGRITY VALIDATION - Part 4: Sample Data Inspection
# =============================================================================

print("\n" + "="*70)
print("VALIDATION PART 4: SAMPLE DATA INSPECTION")
print("="*70 + "\n")

df = spark.table("skyops.raw.bts_ontime")

# 1. Show sample records from each year
print("Sample records from each year:\n")
for year in [2022, 2023, 2024, 2025, 2026]:
    year_sample = df.filter(col("YEAR") == year).limit(2).select(
        "FL_DATE", "AIRLINE_CODE", "FL_NUMBER", "ORIGIN_CITY", "DEST", 
        "DEP_DELAY", "ARR_DELAY", "YEAR", "MONTH"
    )
    print(f"Year {year}:")
    year_sample.show(truncate=False)

print("\n" + "-"*70)

# 2. Check airlines represented
print("\nAirlines in dataset:")
airlines = df.groupBy("AIRLINE_CODE").count().orderBy(col("count").desc())
print(f"Total distinct airlines: {airlines.count()}")
print("\nTop 10 airlines by flight count:")
airlines.show(10, truncate=False)

print("\n" + "-"*70)

# 3. Check busiest routes
print("\nTop 10 busiest routes:")
routes = df.groupBy("ORIGIN_CITY", "DEST_CITY").count() \
    .orderBy(col("count").desc()) \
    .limit(10)
routes.show(truncate=False)

print("\n" + "="*70)
print("\n✓ SAMPLE DATA INSPECTION COMPLETE")
print("  Data appears valid and representative of US flight operations")
print("="*70)

In [0]:
# =============================================================================
# BRONZE LAYER VALIDATION SUMMARY
# =============================================================================

print("\n" + "#"*80)
print("#" + " "*78 + "#")
print("#" + "  BTS BRONZE LAYER INGESTION - VALIDATION REPORT".center(78) + "#")
print("#" + " "*78 + "#")
print("#"*80 + "\n")

print("✅ BRONZE TABLE CREATED SUCCESSFULLY")
print("   Table: skyops.raw.bts_ontime")
print("   Layer: BRONZE (raw ingestion with minimal transformation)")
print("   Type: Managed Delta Table (Unity Catalog)")
print("   Format: Delta Lake with ACID guarantees")
print("   Partition: YEAR (2022-2026)\n")

print("✅ DATA INTEGRITY CONFIRMED")
print("   Source CSV files: 28,178,862 rows")
print("   Delta table rows: 28,178,862 rows")
print("   Data loss: 0 rows (0.0000%)")
print("   Match: PERFECT ✓✓✓\n")

print("✅ SCHEMA VALIDATION PASSED")
print("   Expected columns: 34 (32 business + 2 partition)")
print("   Actual columns: 34")
print("   All columns present and correctly typed\n")

print("✅ DATA QUALITY CHECKS PASSED")
print("   Empty rows: 0")
print("   Year coverage: 2022-2026 (all present)")
print("   Date range: 20220201 to 20260228 (49 months)")
print("   Distinct dates: 1,489 days")
print("   Key columns: <1% null (acceptable for operational data)\n")

print("✅ PARTITION DISTRIBUTION")
print("   2022:  6,191,223 rows (partial - starts Feb)")
print("   2023:  6,847,899 rows (full year)")
print("   2024:  7,079,081 rows (full year + leap day)")
print("   2025:  7,001,619 rows (full year)")
print("   2026:  1,059,040 rows (partial - Jan-Feb only)\n")

print("#"*80)
print("#" + " "*78 + "#")
print("#" + "  ✓ BRONZE LAYER COMPLETE".center(78) + "#")
print("#" + " "*78 + "#")
print("#"*80 + "\n")

print("Medallion Architecture Progress:")
print("  ✓ RAW: BTS TranStats API")
print("  ✓ BRONZE: skyops.raw.bts_ontime (this notebook)")
print("  ◻ SILVER: Data cleansing & quality (next: /SkyOps/silver/)")
print("  ◻ GOLD: Business metrics (future: /SkyOps/gold/)\n")

print("Next Steps:")
print("  1. Create Silver notebook: /SkyOps/silver/bts_ontime_clean.py")
print("  2. Parse FL_DATE to proper date/timestamp types")
print("  3. Add timezone conversions for flight times")
print("  4. Enrich with airport/airline reference data")
print("  5. Apply data quality rules and create SCD tracking\n")

print("Query Bronze table: SELECT * FROM skyops.raw.bts_ontime LIMIT 10")